## **04 - Envoi des CSV vers AWS S3**

Ce notebook envoie les fichiers CSV generes dans `outputs/data` vers un bucket S3.

**Prerequis**:
- Avoir renseigne les variables AWS dans `.env`
- Avoir execute les notebooks precedents pour generer les CSV

In [15]:
import os
from pathlib import Path
from datetime import datetime, timezone

import boto3
import pandas as pd
from botocore.exceptions import ClientError, NoCredentialsError
from dotenv import load_dotenv

In [16]:
# Chargement des variables d'environnement
load_dotenv()

AWS_BUCKET_NAME = os.getenv("AWS_BUCKET_NAME")
AWS_REGION_NAME = os.getenv("AWS_REGION_NAME", "eu-west-3")
AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

required_vars = {
    "AWS_BUCKET_NAME": AWS_BUCKET_NAME,
    "AWS_ACCESS_KEY": AWS_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": AWS_SECRET_ACCESS_KEY,
}

missing = [k for k, v in required_vars.items() if not v]
if missing:
    raise ValueError(f"Variables manquantes dans .env: {missing}")

print("Configuration AWS chargee.")
print(f"Bucket cible: {AWS_BUCKET_NAME}")
print(f"Region: {AWS_REGION_NAME}")

Configuration AWS chargee.
Bucket cible: amzn-jedha-39
Region: eu-west-3


In [17]:
# CSV a envoyer\n
output_data_dir = Path("../outputs/data")
csv_files = sorted(output_data_dir.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Aucun CSV trouve dans ../outputs/data")

print("CSV detectes:")
for f in csv_files:
    print(f"- {f.name}")

csv_files

CSV detectes:
- cities_geocoded.csv
- cities_weather_7d_raw.csv
- cities_weather_scored.csv
- top20_booking_hotels_enriched.csv


[PosixPath('../outputs/data/cities_geocoded.csv'),
 PosixPath('../outputs/data/cities_weather_7d_raw.csv'),
 PosixPath('../outputs/data/cities_weather_scored.csv'),
 PosixPath('../outputs/data/top20_booking_hotels_enriched.csv')]

In [18]:
# Client S3\n
s3_client = boto3.client(
    "s3",
    region_name=AWS_REGION_NAME,
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

# Verification d'acces au bucket
try:
    s3_client.head_bucket(Bucket=AWS_BUCKET_NAME)
    print(f"Acces valide au bucket: {AWS_BUCKET_NAME}")
except ClientError as e:
    raise RuntimeError(f"Impossible d'acceder au bucket '{AWS_BUCKET_NAME}': {e}")

Acces valide au bucket: amzn-jedha-39


In [19]:
def upload_csv_to_s3(local_path: Path, bucket: str, s3_prefix: str = "datasets/kayak") -> dict:
    if not local_path.exists():
        raise FileNotFoundError(f"Fichier introuvable: {local_path}")

    s3_key = f"{s3_prefix}/{local_path.name}"

    s3_client.upload_file(
        Filename=str(local_path),
        Bucket=bucket,
        Key=s3_key,
        ExtraArgs={"ContentType": "text/csv"},
    )

    return {
        "file_name": local_path.name,
        "local_path": str(local_path.resolve()),
        "s3_bucket": bucket,
        "s3_key": s3_key,
        "s3_uri": f"s3://{bucket}/{s3_key}",
        "uploaded_at_utc": datetime.now(timezone.utc).isoformat(),
    }

In [20]:
# Upload de tous les CSV
upload_results = []

for csv_path in csv_files:
    try:
        result = upload_csv_to_s3(csv_path, AWS_BUCKET_NAME)
        upload_results.append(result)
        print(f"OK -> {result['s3_uri']}")
    except (ClientError, NoCredentialsError, FileNotFoundError) as e:
        print(f"ECHEC -> {csv_path.name}: {e}")

uploads_df = pd.DataFrame(upload_results)
uploads_df

OK -> s3://amzn-jedha-39/datasets/kayak/cities_geocoded.csv
OK -> s3://amzn-jedha-39/datasets/kayak/cities_weather_7d_raw.csv
OK -> s3://amzn-jedha-39/datasets/kayak/cities_weather_scored.csv
OK -> s3://amzn-jedha-39/datasets/kayak/top20_booking_hotels_enriched.csv


,file_name,local_path,s3_bucket,s3_key,s3_uri,uploaded_at_utc
0,cities_geocoded.csv,/home/raphael/DataProjects/jedha-certif/projet...,amzn-jedha-39,datasets/kayak/cities_geocoded.csv,s3://amzn-jedha-39/datasets/kayak/cities_geoco...,2026-04-23T03:28:22.286440+00:00
1,cities_weather_7d_raw.csv,/home/raphael/DataProjects/jedha-certif/projet...,amzn-jedha-39,datasets/kayak/cities_weather_7d_raw.csv,s3://amzn-jedha-39/datasets/kayak/cities_weath...,2026-04-23T03:28:22.896053+00:00
2,cities_weather_scored.csv,/home/raphael/DataProjects/jedha-certif/projet...,amzn-jedha-39,datasets/kayak/cities_weather_scored.csv,s3://amzn-jedha-39/datasets/kayak/cities_weath...,2026-04-23T03:28:23.410521+00:00
3,top20_booking_hotels_enriched.csv,/home/raphael/DataProjects/jedha-certif/projet...,amzn-jedha-39,datasets/kayak/top20_booking_hotels_enriched.csv,s3://amzn-jedha-39/datasets/kayak/top20_bookin...,2026-04-23T03:28:24.205534+00:00


In [21]:
# Sauvegarde locale d'un journal d'upload (optionnel)
logs_dir = Path("../outputs/logs")
logs_dir.mkdir(parents=True, exist_ok=True)

log_file = logs_dir / "s3_uploads.csv"
uploads_df.to_csv(log_file, index=False)

print(f"Journal sauvegarde: {log_file.resolve()}")
print(f"Fichiers envoyes: {len(uploads_df)}")

Journal sauvegarde: /home/raphael/DataProjects/jedha-certif/projet-kayak-bloc1/outputs/logs/s3_uploads.csv
Fichiers envoyes: 4
